import numpy as npt

In [1]:
from pubchempy import Compound as c
import numpy as np
import pandas as pd
from pathlib import Path
from rdkit import Chem
import sys
PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

In [6]:
# scrape_data.py

import csv
import logging
from typing import List
import requests


class PubChemSmilesScraper:
    """
    Scrapes SMILES strings from PubChem using the PUG REST API.

    Documentation here
    https://pubchem.ncbi.nlm.nih.gov/docs/pug-rest-tutorial

    Parameters
    ----------
    chunk_size : int
        Number of molecule IDs per API request. keep chunk small to avoidtimeout
    num_chunks : int
        Number of chunks to retrieve.
    """

    BASE_URL = (
        "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/"
        "{}/property/CanonicalSMILES/CSV"
    )

    def __init__(self, chunk_size: int = 200, num_chunks: int = 10):
        self.chunk_size = chunk_size
        self.num_chunks = num_chunks
        self.total_molecules = chunk_size * num_chunks

        logging.basicConfig(level=logging.INFO)
        self.logger = logging.getLogger(self.__class__.__name__)

    def _build_url(self, start_cid: int) -> str:
        """
        Construct API request URL for a chunk of CIDs.
        """
        cid_range = ",".join(
            str(start_cid + i) for i in range(self.chunk_size)
        )
        return self.BASE_URL.format(cid_range)

    def _fetch_chunk(self, start_cid: int):

        cid_list = ",".join(
            str(start_cid + i)
            for i in range(self.chunk_size)
        )

        url = (
            "https://pubchem.ncbi.nlm.nih.gov/rest/pug/"
            "compound/cid/property/CanonicalSMILES/CSV"
        )

        try:
            response = requests.post(
                url,
                data={"cid": cid_list},
                headers={
                    "Content-Type":
                    "application/x-www-form-urlencoded"
                },
                timeout=60
            )

            response.raise_for_status()

        except requests.RequestException as e:
            self.logger.error(
                f"Request failed at CID {start_cid}: {e}"
            )
            return []

        rows = response.text.strip().split("\n")[1:]

        smiles = [
            row[1]
            for row in csv.reader(rows)
            if len(row) > 1
        ]

        return smiles

    def get_data(self) -> List[str]:
        """
        Retrieve SMILES strings across all chunks.
        """
        all_smiles = []

        for chunk_index in range(self.num_chunks):

            start_cid = chunk_index * self.chunk_size + 1

            self.logger.info(
                f"Fetching chunk {chunk_index + 1}/{self.num_chunks} "
                f"(CID {start_cid} → {start_cid + self.chunk_size - 1})"
            )

            smiles_chunk = self._fetch_chunk(start_cid)

            all_smiles.extend(smiles_chunk)

        return all_smiles

    def write_data(self, output_file: str = "pubchem_smiles.csv") -> None:
        """
        Retrieve SMILES data and write to CSV file.

        Parameters
        ----------
        output_file : str
            Output CSV filename.
        """

        smiles_data = self.get_data()

        if not smiles_data:
            self.logger.warning("No SMILES data retrieved.")
            return

        with open(output_file, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["CanonicalSMILES"])

            for smiles in smiles_data:
                writer.writerow([smiles])

        self.logger.info(
            f"Saved {len(smiles_data)} SMILES strings to {output_file}"
        )

In [2]:
# load smiles
from utils.paths import RAW_SMILES
smiles_data = pd.read_csv(RAW_SMILES).values.squeeze()

In [3]:
smiles_data

array(['CC(=O)OC(CC(=O)[O-])C[N+](C)(C)C',
       'CC(=O)OC(CC(=O)O)C[N+](C)(C)C', 'C1=CC(C(C(=C1)C(=O)O)O)O', ...,
       'CC1C=C(C(=O)C2(C1CC3C4(C2C(=O)C(=C(C4CC(=O)O3)C)OC)C)C)OC',
       'C1C(=S)N(C2=C(C=C(C=C2)Cl)C(=N1)C3=CC=CC=C3F)CC(F)(F)F',
       'CC1C(=O)N=C2N1CC3=C(N2)C=CC=C3Cl'], shape=(5000,), dtype=object)

In [6]:
smiles_string = smiles_data[100]
smiles_string = "CCC1234"
mol = Chem.MolFromSmiles(smiles_string)
print(smiles_string)
mol

# filter out the molecules taht have heavy atoms 
# only halogens, O, N, P, S, 
# very lengthy and very short strings will be filtered out later


CCC1234


[13:46:42] SMILES Parse Error: unclosed ring for input: 'CCC1234'


In [7]:
mol is None

True

In [23]:
# pcs = PubChemSmilesScraper(chunk_size=500, num_chunks=10)
# pcs.write_data()

In [8]:
import re
import pickle
from collections import Counter

# all allowed elements 
# will be updated if necessary
ALLOWED_ELEMENTS = {
            "H", "C", "N", "O", "P", "S",
            "F", "Cl", "Br", "I",
            "B", "Si"
        }

class SmilesTokenizer:

    def __init__(self, max_length=128):

        self.max_length = max_length
        self.token_to_idx = None
        self.idx_to_token = None

        self.pattern = r"Cl|Br|\[[^\]]+\]|\.|\=|\#|\-|\+|\\\\|\/|\(|\)|[A-Za-z]|[0-9]"


    def contains_only_allowed_atoms(self, smiles):

        bracket_atoms = re.findall(r"\[([A-Z][a-z]?)", smiles)
        simple_atoms = re.findall(r"Cl|Br|[A-Z][a-z]?", smiles)

        atoms = set(bracket_atoms + simple_atoms)

        return atoms.issubset(ALLOWED_ELEMENTS)


    def contains_disallowed_bracket_atoms(self, smiles) -> bool :
        '''
        Checks whether the smiles string contains disallowed characters

        Returns True or False.
        '''

        bracket_tokens = re.findall(r"\[[^\]]+\]", smiles)

        for token in bracket_tokens:
            if token not in ALLOWED_ELEMENTS:
                return True

        return False
    

    def contains_charge(self, smiles):

        charged_atoms = re.findall(r"\[[^\]]*[\+\-][^\]]*\]", smiles)

        return len(charged_atoms) > 0


    def filter_smiles(self, smiles_list):
        
        '''
        Filters the given SMILES based on some conditions
        
        More conditions can be added or removed 
        '''

        filtered = [
            smi for smi in smiles_list
            if self.contains_only_allowed_atoms(smi)
            and not self.contains_charge(smi)
            and not self.contains_disallowed_bracket_atoms(smi)
            and "." not in smi
        ]

        return filtered
    



    def tokenize(self, smiles):

        tokens = re.findall(self.pattern, smiles)

        return ["<START>"] + tokens + ["<END>"]


    def fit(self, smiles_list):
        '''
        Createa a vocab and inverse vocab 
        '''

        smiles_list = self.filter_smiles(smiles_list)

        tokenized = [self.tokenize(s) for s in smiles_list]

        vocab = Counter()

        for seq in tokenized:
            vocab.update(seq)

        vocab_list = ["<PAD>", "<UNK>"] + sorted(vocab.keys())

        self.token_to_idx = {
            token: idx for idx, token in enumerate(vocab_list)
        }

        self.idx_to_token = {
            idx: token for token, idx in self.token_to_idx.items()
        }


    def encode(self, smiles):

        tokens = self.tokenize(smiles)

        encoded = [
            self.token_to_idx.get(token, self.token_to_idx["<UNK>"])
            for token in tokens
            ]

        encoded = encoded[:self.max_length]

        encoded += [
            self.token_to_idx["<PAD>"]
        ] * (self.max_length - len(encoded))

        return encoded

    def encode_batch(self, smiles_list):
        return [
            self.encode(smi)
            for smi in smiles_list
        ]


    def decode(self, indices):

        tokens = []

        for i in indices:

            token = self.idx_to_token.get(i, "<UNK>")

            if token == "<END>":
                break

            if token != "<PAD>":
                tokens.append(token)

        return tokens

    def decode_batch(self, indices_list):
        return [
            self.decode(indices)
            for indices in indices_list
        ]

    def detokenize(self, tokens):
        '''
        Remove the token characters
        '''

        return "".join(
            t for t in tokens
            if t not in {"<START>", "<END>"}
        )


    def save(self, filepath):

        with open(filepath, "wb") as f:

            pickle.dump({
                "token_to_idx": self.token_to_idx,
                "idx_to_token": self.idx_to_token
            }, f)


    def load(self, filepath):

        with open(filepath, "rb") as f:

            data = pickle.load(f)

        self.token_to_idx = data["token_to_idx"]
        self.idx_to_token = data["idx_to_token"]

In [ ]:
t = SmilesTokenizer()
t.load("vocab.pkl")
tokens = t.encode_batch(smiles_data[:10])
smiles = [t.detokenize(t.decode(token)) for token in tokens]
smiles

['CC(=O)OC(CC(=O)<UNK>)C<UNK>(C)(C)C',
 'CC(=O)OC(CC(=O)O)C<UNK>(C)(C)C',
 'C1=CC(C(C(=C1)C(=O)O)O)O',
 'CC(CN)O',
 'C(C(=O)COP(=O)(O)O)N',
 'C1=CC(=C(C=C1<UNK>(=O)<UNK>)<UNK>(=O)<UNK>)Cl',
 'CCN1C=NC2=C(N=CN=C21)N',
 'CCC(C)(C(C(=O)O)O)O',
 'C1(C(C(C(C(C1O)O)OP(=O)(O)O)O)O)O',
 'C1C(NC2=C(N1)NC(=NC2=O)N)CN(C=O)C3=CC=C(C=C3)C(=O)NC(CCC(=O)O)C(=O)O']

In [30]:
import torch
encoded_sequences = torch.load(PROJECT_ROOT/"artifacts/encoded_dataset.pt")
len(encoded_sequences[0])

128

In [11]:
import yaml

with open(PROJECT_ROOT/ "configs/config.yaml") as f:
    config = yaml.safe_load(f)

In [12]:
tokenizer = SmilesTokenizer()
tokenizer.load(PROJECT_ROOT / "artifacts/vocab.pkl")
vocab_size = len(tokenizer.token_to_idx)
print(vocab_size)

27


In [13]:
config

{'model': {'vocab_size': None,
  'embedding_dim': 128,
  'hidden_dim': 256,
  'num_layers': 2,
  'dropout': 0.2},
 'training': {'batch_size': 256, 'learning_rate': 0.001},
 'sampling': {'block_size': 128}}

In [ ]:
import torch
import torch.nn as nn


class LSTM(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers):

        super().__init__()

        self.hidden_dim = hidden_size
        self.num_layers = num_layers

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_size,
            num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, vocab_size)


    def forward(self, x):

        x = self.embedding(x)

        out, _ = self.lstm(x)

        out = self.fc(out)

        return out


    def generate(self, idx, max_new_tokens, block_size):

        for _ in range(max_new_tokens):

            idx_cond = idx[:, -block_size:]

            logits = self(idx_cond)

            logits = logits[:, -1, :]

            probs = torch.softmax(logits, dim=-1)

            idx_next = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, idx_next), dim=1)

        return idx

In [15]:
from models.lstm_model import LSTM
model = LSTM(
    vocab_size=vocab_size,
    embedding_dim=config["model"]["embedding_dim"],
    hidden_size=config["model"]["hidden_dim"],
    num_layers=config["model"]["num_layers"]
)

In [ ]:
# trainer.py

import torch
from model import Transformer
from preprocessing import Tokenizer

class Trainer:
    def __init__(self):
      self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
      print("\nUsing device:", self.device)


      self.model = Transformer(input_channels=1, latent_dim=16).to(self.device)

      t = Tokenizer()
      train_ds, val_ds = t.create_train_val("smiles_list.csv", 
                                            input_seq_length=20, 
                                            train_size=0.8)

      self.train_dataloader = torch.utils.data.DataLoader(train_ds, batch_size=64, shuffle=True)
      self.val_dataloader = torch.utils.data.DataLoader(val_ds, batch_size=64)

      self.epoch_list = []
      self.loss_list = []


    def train(self, epochs, lr):

        # Implement training logic here
        print(f"\nTraining model on {self.device}...")
        optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)
        # optimizer = torch.optim.SGD(self.model.parameters(), lr=lr)
        
        self.model.to(self.device)

        for epoch in range(epochs):
          total_loss = 0
          for i, (images, labels) in enumerate(self.train_dataloader):
              images, labels = images.to(self.device), labels.to(self.device)
              # Forward pass
              outputs, mu, logvar = self.model(images)
              # loss = criterion(outputs, labels)
              loss = self.loss_function(images, outputs, mu, logvar)
    
              # Backward and optimize
              optimizer.zero_grad()
              loss.backward()
              optimizer.step()

          total_loss = +loss.item()
          self.epoch_list.append(epoch)
          self.loss_list.append(total_loss)

          print(f'Epoch [{epoch+1}/{epochs}],Train loss: {total_loss/64:.6f}')

    def loss_function(self,x, x_hat, mu, logvar):
      BCE = torch.nn.functional.binary_cross_entropy(x_hat, x, reduction='sum')
      KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
      return BCE + KLD

    def plot_training(self):
      import matplotlib.pyplot as plt
      # plot the epoch vs loss
      plt.plot(self.epoch_list, self.loss_list)
      plt.xlabel('Epoch')
      plt.ylabel('Loss')
      plt.title('Training Loss')
      plt.show()




{}